# A Backtracking Solver for CSPs

This notebook implements a variant of the backtracking solver from the notebook `02-Backtracking-Constraint-Solver.ipynb`.  This variant counts the number of partial assignments that are tested and calls a function `onUpdate` for every partial assignment that is considered.  This function can be used to visualize the search.  The code in this notebook is also contained in the file `Backtrack-Solver-Animate.jsh`.

We use the class `RecursiveSet` and the function `compileConstraints` from the file `ConstraintCompiler.jsh`, which has been discussed in the notebook `02-Backtracking-Constraint-Solver.ipynb`.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load ConstraintCompiler.jsh

In [ ]:
record CSP(List<String> variables, List<Integer> values, List<String> constraints) {}

// A constraint together with the set of its variables and a compiled predicate that
// evaluates the constraint for a given variable assignment.
record AnnotatedConstraint(String formula,
                           RecursiveSet<String> variables,
                           Predicate<Map<String, Integer>> predicate) {}

// The result of a successful search: the number of steps and the solution.
record SolveResult(int steps, Map<String, Integer> solution) {}

// The state of the search.  It counts the number of partial assignments that have been tested.
class SearchState {
    int steps = 0;
}

The function `collectVariables(expr)` takes a string `expr` that can be interpreted as a valid expression as its input and collects all variables occurring in `expr`.
It takes care to remove names that correspond to predefined values or functions, i.e. the names `Math`, `true`, and `false` are not treated as variables.

In [ ]:
// Returns the set of variables occurring in the expression expr.
RecursiveSet<String> collectVariables(String expr) {
    Pattern identifier = Pattern.compile("(?<!\\.)\\b[a-zA-Z_][a-zA-Z0-9_]*\\b");
    Matcher matcher    = identifier.matcher(expr);
    RecursiveSet<String> variables = new RecursiveSet<>();
    while (matcher.find()) {
        String candidate = matcher.group();
        if (!Set.of("Math", "true", "false").contains(candidate)) {
            variables.add(candidate);
        }
    }
    return variables;
}

In [ ]:
var expr = "Math.abs(x - y) + Math.abs(z1 - z2)";
collectVariables(expr)

## The Backtracking Solver with Animation

The function `isConsistent(variable, value, assignment, constraints)` takes four arguments:
- `variable` is a variable that does not occur in `assignment`,
- `value` is a value that can be substituted for this variable,
- `assignment` is a *consistent* partial variable assignment.
- `constraints` is a list of triples of the form $\langle f, V, P \rangle$ where $f$ is a formula, $V$ is the set of variables occurring in $f$, and $P$ is the compiled predicate that evaluates $f$.

This function returns `true` iff the partial variable assignment
$$\texttt{assignment} \cup \bigl\{\langle\texttt{variable} \mapsto\texttt{value}\rangle\bigr\}$$
is consistent with all the constraints $f$ occurring in `constraints`.

**Note** that the predicate $P$ handles the variable substitution dynamically.

In [ ]:
// Checks whether the assignment assignment ∪ {variable ↦ value} is consistent with
// all constraints that can be evaluated using this assignment.
boolean isConsistent(String variable, int value,
                     Map<String, Integer> assignment,
                     List<AnnotatedConstraint> constraints) {
    var newAssignment = new HashMap<>(assignment);
    newAssignment.put(variable, value);
    var assignedVars = newAssignment.keySet();
    for (var c : constraints) {
        if (c.variables().contains(variable) && c.variables().isSubset(assignedVars)) {
            if (!c.predicate().test(newAssignment)) {
                return false;
            }
        }
    }
    return true;
}

The function `isComplete` checks whether the given assignment assigns a value to every variable.

In [ ]:
// Checks whether every variable has been assigned a value.
boolean isComplete(Map<String, Integer> assignment, List<String> variables) {
    return assignment.keySet().containsAll(variables);
}

The function `backtrackSearch` takes four arguments:
- `assignment` is a partial variable assignment that is represented as a `Map`. Initially, this assignment will be the empty map. Every recursive call of `backtrackSearch` adds the assignment of one variable to the given assignment. The important invariant of recursive calls of `backtrackSearch` is that `assignment` is *consistent*, i.e. all constraints $f$ that contain only variables from the set $\mathtt{dom}(\mathtt{assignment})$ are satisfied.
- `csp` is an *augmented constraint satisfaction problem*,
  i.e. `csp` is a triple of the form
  $$ \mathcal{P} = \langle \mathtt{Vars}, \mathtt{Values}, \mathtt{Constraints} \rangle $$
  where
  - $\mathtt{Vars}$ is a set of strings which serve as *variables*,
  - $\mathtt{Values}$ is a set of *values* that can be assigned
    to the variables in $\mathtt{Vars}$.
  - $\mathtt{Constraints}$ is a list of triples of the form $(f, V, P)$ where $f$ is a Boolean expression, $V$ is the set of variables occurring in $f$, and $P$ is the compiled predicate.
- `state` is an object tracking the number of steps.
- `onUpdate` is a callback function for visualization.  It is an object of type `Consumer<Map<String, Integer>>`.  If it is `null`, it is not called.

The function `backtrackSearch` tries to find a solution of `csp` by recursively augmenting `assignment`.

In [ ]:
// Tries to extend the consistent partial assignment to a solution of the CSP.
Map<String, Integer> backtrackSearch(Map<String, Integer> assignment,
                                     List<String> variables,
                                     List<Integer> values,
                                     List<AnnotatedConstraint> constraints,
                                     SearchState state,
                                     Consumer<Map<String, Integer>> onUpdate) {
    if (onUpdate != null) {
        onUpdate.accept(assignment);
    }
    if (isComplete(assignment, variables)) {
        return assignment;
    }
    String unassignedVar = variables.stream().filter(v -> !assignment.containsKey(v)).findFirst().get();
    for (int value : values) {
        state.steps++;
        if (isConsistent(unassignedVar, value, assignment, constraints)) {
            var newAssignment = new LinkedHashMap<>(assignment);
            newAssignment.put(unassignedVar, value);
            var result = backtrackSearch(newAssignment, variables, values, constraints, state, onUpdate);
            if (result != null) {
                return result;
            }
        }
    }
    return null;
}

The procedure `solve(csp)` takes a *constraint satisfaction problem*
`csp` as input. Here `csp` is a triple of the form
$$ \mathcal{P} = \langle \mathtt{Variables}, \mathtt{Values}, \mathtt{Constraints} \rangle $$
where
- `variables` is a set of strings which serve as *variables*,
- `values` is a set of *values* that can be assigned
  to the variables in the set `variables`.
- `constraints` is a list of formulas (constraints).

The main purpose of the function `solve` is to convert the CSP `csp` into an
*augmented CSP* where every constraint $f$ is annotated with
the variables occurring in $f$. This annotated CSP is then solved using the function
`backtrackSearch`.

As Java does not support optional parameters, there is a second version of `solve` that takes only one parameter.  This version does not visualize the search.

In [ ]:
// Solves the given CSP.  The function onUpdate is called with every partial
// assignment that is considered.  Returns null if the CSP is not solvable.
SolveResult solve(CSP csp, Consumer<Map<String, Integer>> onUpdate) {
    var state      = new SearchState();
    var Variables  = csp.constraints().stream().map(f -> collectVariables(f)).toList();
    var predicates = compileConstraints(csp.constraints(), Variables);
    List<AnnotatedConstraint> annotatedConstraints = new ArrayList<>();
    for (int i = 0; i < csp.constraints().size(); ++i) {
        annotatedConstraints.add(new AnnotatedConstraint(csp.constraints().get(i),
                                                         Variables.get(i),
                                                         predicates.get(i)));
    }
    var solution = backtrackSearch(new LinkedHashMap<>(), csp.variables(), csp.values(),
                                   annotatedConstraints, state, onUpdate);
    System.out.println("Tested " + state.steps + " partial assignments");
    if (solution == null) {
        return null;
    }
    return new SolveResult(state.steps, solution);
}

SolveResult solve(CSP csp) {
    return solve(csp, null);
}

In [ ]:
var csp = new CSP(List.of("x", "y", "z"),
                  List.of(1, 2, 3, 4, 5),
                  List.of("x * y == z + 1", "x + y == z", "x * y == 2 * z - x * x"));
solve(csp, assignment -> System.out.println(assignment))